# Stage 1b — Labeling Pilot

**AAI-590 Capstone · Monish Yarapathineni**

Applies the 8-category taxonomy from Stage 1a to a stratified sample of real
FoundationalASSIST wrong answers, before committing to the full 20,520-pair run.

The taxonomy was derived from the Eedi corpus — UK curriculum, diagnostic
multiple-choice, ages 7–18. FoundationalASSIST is US Common Core, grades 6–8,
and 65% fill-in-the-blank. This pilot measures whether the categories survive
that shift.

### Four questions this notebook answers

| # | Question | Why it matters |
|---|---|---|
| 1 | Which categories transfer, and at what rate? | Determines the classifier's output dimension |
| 2 | Do fill-in and multiple-choice differ in labelability? | Separates *rare* from *undetectable* |
| 3 | How often does the model agree with itself? | Label noise caps achievable model performance |
| 4 | What share is unassignable? | Reveals genuine gaps in the taxonomy |

**Question 2 is the important one.** A category can be absent from the labels for
two very different reasons: the underlying error does not occur in this content
(*rare*), or it occurs but the answer format cannot reveal it (*undetectable*).
A bare numeric answer like `12` cannot expose a vocabulary misconception the way
a written distractor can. Only the second case is a measurement limitation, and
conflating them would produce false claims about students in the report.

---

## 1 · Setup

In [ ]:
try:
    import google.colab
    IN_COLAB = True
    %pip install -q datasets huggingface_hub anthropic
except ImportError:
    IN_COLAB = False

import os, re, json, time, textwrap, random
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import load_dataset, load_from_disk
from anthropic import Anthropic

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.05)
plt.rcParams.update({'figure.dpi': 130, 'figure.figsize': (9, 4)})

RANDOM_SEED = 590
random.seed(RANDOM_SEED); np.random.seed(RANDOM_SEED)

print(f'Running on Colab: {IN_COLAB}')

## 2 · Load the taxonomy

Upload `misconception_taxonomy_v1.json` from Stage 1a, or point `TAXONOMY_PATH`
at it if you have it in Drive.

In [ ]:
TAXONOMY_PATH = "misconception_taxonomy_v1.json"

if not os.path.exists(TAXONOMY_PATH) and IN_COLAB:
    from google.colab import files
    print("Upload misconception_taxonomy_v1.json:")
    up = files.upload()
    TAXONOMY_PATH = list(up.keys())[0]

with open(TAXONOMY_PATH) as f:
    artifact = json.load(f)

taxonomy   = artifact["taxonomy"]
categories = taxonomy["categories"]
category_names = [c["name"] for c in categories]

# Eedi baseline distribution, for comparison later
eedi_dist = artifact["validation"]["distribution"]
eedi_total = sum(v for k, v in eedi_dist.items() if k != "UNASSIGNABLE")
eedi_share = {k: v / eedi_total for k, v in eedi_dist.items() if k != "UNASSIGNABLE"}

print(f"Taxonomy: {len(category_names)} categories "
      f"(generated {artifact['generated_utc']}, model {artifact['taxonomy_model']})\n")
for n in category_names:
    print(f"  • {n:<38} Eedi baseline {eedi_share.get(n, 0):>6.1%}")

## 3 · Load FoundationalASSIST

In [ ]:
if IN_COLAB:
    from huggingface_hub import login
    login()

HF_REPO = 'ASSISTments/FoundationalASSIST'
LOCAL_BASE = os.path.expanduser(
    '~/Desktop/Educator/aai590-capstone/data/foundationalassist'
)

if IN_COLAB:
    problems_ds     = load_dataset(HF_REPO, 'Foundational ASSIST Dataset')
    interactions_ds = load_dataset(HF_REPO, 'Interactions')
else:
    problems_ds     = load_from_disk(os.path.join(LOCAL_BASE, 'Foundational ASSIST Dataset'))
    interactions_ds = load_from_disk(os.path.join(LOCAL_BASE, 'interactions'))

problems     = problems_ds['train'].to_pandas()
interactions = interactions_ds['train'].to_pandas()

print(f'Problems:     {problems.shape}')
print(f'Interactions: {interactions.shape}')

## 4 · Rebuild the misconception rows

Same definition established in notebook 01 §10: score of zero **and** the
submitted answer differs from the correct one. Rows where the answer matches but
the score is zero are hint-penalty rows, which carry behavioural signal but no
misconception.

In [ ]:
def strip_html(text):
    """Strip HTML tags, MathML markup, and common HTML entities."""
    if pd.isna(text):
        return ""
    text = str(text)
    text = re.sub(r'xmlns="[^"]*"', '', text)
    text = re.sub(r'<mfrac>\s*<mn>([^<]+)</mn>\s*<mn>([^<]+)</mn>\s*</mfrac>', r'\1/\2', text)
    text = re.sub(r'<msup>\s*<mi>([^<]+)</mi>\s*<mn>([^<]+)</mn>\s*</msup>', r'\1^\2', text)
    text = re.sub(r'<msup>\s*<mn>([^<]+)</mn>\s*<mn>([^<]+)</mn>\s*</msup>', r'\1^\2', text)
    text = re.sub(r'<mo>([^<]+)</mo>', r' \1 ', text)
    text = re.sub(r'<m[a-z]+>([^<]*)</m[a-z]+>', r'\1', text)
    text = re.sub(r'<[^>]+>', ' ', text)
    for ent, char in {'&nbsp;': ' ', '&lt;': '<', '&gt;': '>', '&amp;': '&',
                      '&le;': '≤', '&ge;': '≥', '&deg;': '°', '&times;': '×'}.items():
        text = text.replace(ent, char)
    text = re.sub(r'&#\d+;', '', text)
    text = re.sub(r'&[a-z]+;', '', text)
    return re.sub(r'\s+', ' ', text).strip()


def normalise_answer(s):
    if pd.isna(s):
        return ""
    return str(s).strip().lower()


merged = interactions.merge(
    problems[['problem_id', 'Problem Type', 'Answer Types',
              'Fill-in Answers', 'Multiple Choice Answers',
              'Multiple Choice Options', 'Problem Body']],
    on='problem_id', how='left'
)

fillin_mask = merged['Problem Type'] == 'Fill-in-the-blank(s)'
mc_mask     = merged['Problem Type'].isin(
    ['Multiple Choice (select 1)', 'Multiple Choice (select all)']
)

fillin_wrong = merged[
    fillin_mask & (merged['discrete_score'] == 0) &
    merged.apply(lambda r: normalise_answer(r['answer_text'])
                 != normalise_answer(r['Fill-in Answers']), axis=1)
]
mc_wrong = merged[
    mc_mask & (merged['discrete_score'] == 0) &
    merged.apply(lambda r: normalise_answer(r['answer_text'])
                 != normalise_answer(r['Multiple Choice Answers']), axis=1)
]

print(f'Fill-in misconception rows : {len(fillin_wrong):,}')
print(f'MC misconception rows      : {len(mc_wrong):,}')
print(f'Total                      : {len(fillin_wrong) + len(mc_wrong):,}')

## 5 · Build the labeling triples

The unit of labeling is a unique **(problem, wrong answer)** pair, not an
individual interaction — the same wrong answer given by 400 students is one
labeling call, and the label joins back onto all 400 rows.

Fill-in pairs are restricted to the **top 10 most common wrong answers per
problem**, the threshold established in notebook 01 §10.5 (≈74% coverage of
wrong attempts).

In [ ]:
TOP_N_FILLIN = 10

def norm_answer_key(s):
    """Collapse delimiter whitespace so identical selections group together."""
    if pd.isna(s):
        return ""
    s = re.sub(r'\s*,\s*', ' , ', str(s).strip())
    return re.sub(r'\s+', ' ', s)

# ---- Fill-in: top N wrong answers per problem -------------------------------
fillin_wrong = fillin_wrong.copy()
fillin_wrong['answer_key'] = fillin_wrong['answer_text'].map(norm_answer_key)

fillin_pairs = (
    fillin_wrong.groupby(['problem_id', 'answer_key'])
    .size().reset_index(name='n_students')
    .rename(columns={'answer_key': 'answer_text'})
)
fillin_pairs['rank'] = (
    fillin_pairs.groupby('problem_id')['n_students']
    .rank(method='first', ascending=False)
)
fillin_pairs = fillin_pairs[fillin_pairs['rank'] <= TOP_N_FILLIN]

# ---- MC: SELECT-1 ONLY ------------------------------------------------------
# Select-all is excluded from labeling. A wrong subset can encode several
# distinct errors simultaneously (an option wrongly included AND an option
# wrongly omitted), so it cannot carry a single misconception label. Those
# 119,714 rows are masked in the Stage 2 loss, not discarded — they remain in
# the student sequence as behavioural context.
mc1_wrong = mc_wrong[mc_wrong['Problem Type'] == 'Multiple Choice (select 1)'].copy()
mc1_wrong['answer_key'] = mc1_wrong['answer_text'].map(norm_answer_key)

mc_pairs = (
    mc1_wrong.groupby(['problem_id', 'answer_key'])
    .size().reset_index(name='n_students')
    .rename(columns={'answer_key': 'answer_text'})
)

n_excluded = (mc_wrong['Problem Type'] == 'Multiple Choice (select all)').sum()
print(f"Select-all rows excluded from labeling : {n_excluded:,} "
      f"({n_excluded / 605_773:.1%} of misconception rows — masked in Stage 2 loss)")
print(f"Fill-in pairs : {len(fillin_pairs):,}")
print(f"MC pairs      : {len(mc_pairs):,}")
print(f"TOTAL         : {len(fillin_pairs) + len(mc_pairs):,}   (design doc: ~20,520)")

# ---- Attach problem context -------------------------------------------------
pmeta = problems.set_index('problem_id')

def build_triples(pairs, fmt):
    rows = []
    for _, r in pairs.iterrows():
        try:
            p = pmeta.loc[r['problem_id']]
        except KeyError:
            continue
        if isinstance(p, pd.DataFrame):
            p = p.iloc[0]
        correct = (p['Fill-in Answers'] if fmt == 'fill-in'
                   else p['Multiple Choice Answers'])
        body = strip_html(p['Problem Body'])
        if not body or pd.isna(correct):
            continue
        rows.append({
            'problem_id':   r['problem_id'],
            'format':       fmt,
            'problem_text': body,
            'correct':      strip_html(correct),
            'wrong':        strip_html(r['answer_text']),
            'options':      strip_html(p['Multiple Choice Options']) if fmt == 'mc' else '',
            'n_students':   r['n_students'],
        })
    return pd.DataFrame(rows)

fillin_triples = build_triples(fillin_pairs, 'fill-in')
mc_triples     = build_triples(mc_pairs, 'mc')

print(f"\nFill-in triples : {len(fillin_triples):,}")
print(f"MC triples      : {len(mc_triples):,}")
print(f"TOTAL           : {len(fillin_triples) + len(mc_triples):,}")

### Stratified pilot sample

**100 fill-in and 100 multiple-choice**, rather than the natural 65/35 split.
The comparison of interest is *between* formats, so equal sample sizes give
comparable precision on both sides. Production labeling runs at natural
proportions; this is a measurement design choice for the pilot only.

In [ ]:
N_PER_FORMAT = 100

pilot = pd.concat([
    fillin_triples.sample(min(N_PER_FORMAT, len(fillin_triples)), random_state=RANDOM_SEED),
    mc_triples.sample(min(N_PER_FORMAT, len(mc_triples)), random_state=RANDOM_SEED),
]).reset_index(drop=True)

print(f'Pilot sample: {len(pilot)} triples')
print(pilot['format'].value_counts().to_string())
print('\n' + '=' * 76)
print('SAMPLE OF WHAT THE MODEL WILL SEE')
print('=' * 76)
for _, r in pilot.groupby('format').head(2).iterrows():
    print(f"\n[{r['format']}]  problem {r['problem_id']}  ({r['n_students']} students)")
    print(f"  Q       : {textwrap.shorten(r['problem_text'], 150)}")
    if r['options']:
        print(f"  Options : {textwrap.shorten(r['options'], 120)}")
    print(f"  Correct : {textwrap.shorten(str(r['correct']), 60)}")
    print(f"  Student : {textwrap.shorten(str(r['wrong']), 60)}")

## 6 · Configure the client

Note the absence of a `temperature` parameter — it is deprecated on current
models, so sampling defaults apply and labeling is **not** deterministic. §8
measures the resulting disagreement rate directly rather than assuming it away.

In [ ]:
try:
    from google.colab import userdata
    api_key = userdata.get("ANTHROPIC_API_KEY")
except ImportError:
    api_key = os.environ.get("ANTHROPIC_API_KEY")

assert api_key, "ANTHROPIC_API_KEY not found — add it to Colab secrets."
client = Anthropic(api_key=api_key)

LABELING_MODEL = "claude-sonnet-5"


def response_text(resp):
    """Concatenate text blocks, skipping thinking blocks."""
    parts = [b.text for b in resp.content if getattr(b, "type", None) == "text"]
    if not parts:
        raise ValueError("No text block; got "
                         f"{[getattr(b, 'type', '?') for b in resp.content]}")
    return "".join(parts)


def extract_json(text):
    candidate = None
    fenced = re.search(r"```(?:json)?\s*(.*?)\s*```", text, re.S)
    if fenced:
        candidate = fenced.group(1)
        try:
            return json.loads(candidate)
        except json.JSONDecodeError:
            candidate = None
    if candidate is None:
        try:
            candidate = text[text.index("{"):text.rindex("}") + 1]
        except ValueError:
            raise ValueError(f"No JSON found:\n{text[:600]}")
    try:
        return json.loads(candidate)
    except json.JSONDecodeError:
        return json.loads(re.sub(r"}\s*{", "},{", candidate))


print("Client ready. Model:", LABELING_MODEL)

## 7 · Label the pilot sample

Each item returns a category, a **confidence rating**, and a one-clause
justification.

Confidence is the instrument for Question 2. If fill-in items draw systematically
lower confidence than multiple-choice items, that is the detectability artifact
made measurable — the model is being asked to infer a misconception type from a
bare numeric string with insufficient signal to do so.

In [ ]:
LABEL_TEMPLATE = """You are labeling student wrong answers with the type of \
misconception they reveal.

CATEGORIES:
{cats}

For each numbered item you are given the problem, the correct answer, and what \
the student actually submitted. Decide which category the student's error \
reveals.

Rules:
- Choose exactly one category, or "UNASSIGNABLE" if none genuinely fits.
- Rate your confidence: "high", "medium", or "low".
- Use "low" when the submitted answer does not carry enough information to \
distinguish between categories — for example, a bare number that could result \
from several different errors. Do not guess confidently.
- Give a one-clause justification (under 15 words).

Return ONLY a JSON object:
{{"1": {{"category": "<name>", "confidence": "<high|medium|low>", "why": "<clause>"}}, ...}}

ITEMS:
{items}"""

cats_block = "\n".join(f"- {c['name']}: {c['definition']}" for c in categories)


def format_item(i, r):
    lines = [f"{i}. PROBLEM: {textwrap.shorten(r['problem_text'], 400)}"]
    if r["options"]:
        lines.append(f"   OPTIONS: {textwrap.shorten(r['options'], 300)}")
    lines.append(f"   CORRECT ANSWER: {r['correct']}")
    lines.append(f"   STUDENT SUBMITTED: {r['wrong']}")
    return "\n".join(lines)


def label_batch(df_batch, start_idx, max_retries=3):
    items = "\n\n".join(
        format_item(start_idx + i + 1, r) for i, (_, r) in enumerate(df_batch.iterrows())
    )
    last = None
    for attempt in range(max_retries):
        try:
            resp = client.messages.create(
                model=LABELING_MODEL,
                max_tokens=16000,
                messages=[{"role": "user", "content":
                           LABEL_TEMPLATE.format(cats=cats_block, items=items)}],
            )
            return extract_json(response_text(resp))
        except Exception as e:
            last = e
            time.sleep(2 ** attempt)
    raise last


def run_labeling_pass(df, batch_size=25, tag=""):
    out, failed = {}, []
    for start in range(0, len(df), batch_size):
        chunk = df.iloc[start:start + batch_size]
        try:
            out.update(label_batch(chunk, start))
        except Exception as e:
            failed.append(start)
            print(f"\n  {tag} batch {start} failed: {e}")
        print(f"\r  {tag} labeled {len(out):,} / {len(df):,}", end="")
    print()
    return out, failed


print("Pass 1 of 2")
pass1, failed1 = run_labeling_pass(pilot, tag="[pass1]")
print(f"  complete — {len(pass1)} labels, {len(failed1)} failed batches")

In [ ]:
# Attach pass-1 results
pilot['category']   = [pass1.get(str(i + 1), {}).get('category', 'MISSING') for i in range(len(pilot))]
pilot['confidence'] = [pass1.get(str(i + 1), {}).get('confidence', 'MISSING') for i in range(len(pilot))]
pilot['why']        = [pass1.get(str(i + 1), {}).get('why', '') for i in range(len(pilot))]

print(pilot[['format', 'category', 'confidence']].head(12).to_string())

## 8 · Question 3 — self-agreement

A second independent pass over the identical items. Disagreement between the two
is label noise that will be baked into the LSTM's training targets, and it places
a ceiling on achievable model performance: the classifier cannot be more reliable
than the labels it learns from.

In [ ]:
print("Pass 2 of 2")
pass2, failed2 = run_labeling_pass(pilot, tag="[pass2]")

pilot['category_p2'] = [pass2.get(str(i + 1), {}).get('category', 'MISSING') for i in range(len(pilot))]

valid = pilot[(pilot['category'] != 'MISSING') & (pilot['category_p2'] != 'MISSING')]
agree = (valid['category'] == valid['category_p2']).mean()

print(f"\n{'=' * 60}")
print(f"SELF-AGREEMENT : {agree:.1%}  (n = {len(valid)})")
print('=' * 60)

for fmt in ['fill-in', 'mc']:
    sub = valid[valid['format'] == fmt]
    if len(sub):
        print(f"  {fmt:<10} {(sub['category'] == sub['category_p2']).mean():.1%}  (n={len(sub)})")

print("\nMost common disagreements:")
dis = valid[valid['category'] != valid['category_p2']]
for (a, b), n in Counter(zip(dis['category'], dis['category_p2'])).most_common(6):
    print(f"  {n:>3}x  {a[:30]:<32} → {b[:30]}")

## 9 · Questions 1 & 4 — transfer and coverage

How the category distribution on real FoundationalASSIST data compares with the
Eedi baseline the taxonomy was derived from.

In [ ]:
labeled = pilot[pilot['category'] != 'MISSING']
dist = Counter(labeled['category'])
n = len(labeled)

rows = []
for name in category_names:
    fa = dist.get(name, 0) / n
    ee = eedi_share.get(name, 0)
    rows.append({'Category': name, 'Eedi': ee, 'FoundationalASSIST': fa, 'Shift': fa - ee})

comp = pd.DataFrame(rows).sort_values('FoundationalASSIST', ascending=False)

print('TRANSFER FROM EEDI → FOUNDATIONALASSIST')
print('=' * 76)
for _, r in comp.iterrows():
    bar = '█' * int(30 * r['FoundationalASSIST'])
    flag = '  ⚠ COLLAPSED' if r['FoundationalASSIST'] < 0.02 else ''
    print(f"{r['Category'][:32]:<34} {r['Eedi']:>6.1%} → {r['FoundationalASSIST']:>6.1%} "
          f"({r['Shift']:+.1%}) {bar}{flag}")

unassign = dist.get('UNASSIGNABLE', 0) / n
print('\n' + '=' * 76)
print(f"Unassignable      : {unassign:>6.1%}   "
      f"{'PASS' if unassign < 0.05 else 'REVIEW — taxonomy gap on this data'}")
print(f"Categories in use : {sum(1 for c in category_names if dist.get(c, 0) > 0):>6} of {len(category_names)}")
collapsed = [c for c in category_names if dist.get(c, 0) / n < 0.02]
if collapsed:
    print(f"Below 2% floor    : {len(collapsed)} → {collapsed}")

## 10 · Question 2 — rare versus undetectable

The distinction that determines what the report may legitimately claim.

A category can be absent because the error does not occur in this content
(**rare** — a real property of grades 6–8 material) or because the answer format
cannot reveal it (**undetectable** — a limitation of the measurement).

Confidence separates them. A category that is rare but *detectable when present*
still draws high confidence on the few instances that occur. A category that is
undetectable produces low confidence wherever it is assigned, because the model
is guessing from insufficient signal.

In [ ]:
fmt_conf = pd.crosstab(labeled['format'], labeled['confidence'], normalize='index')
print('CONFIDENCE BY ANSWER FORMAT')
print('=' * 60)
print((fmt_conf * 100).round(1).to_string())

low_fill = (labeled[labeled['format'] == 'fill-in']['confidence'] == 'low').mean()
low_mc   = (labeled[labeled['format'] == 'mc']['confidence'] == 'low').mean()

print(f"\nLow-confidence share — fill-in : {low_fill:.1%}")
print(f"Low-confidence share — MC      : {low_mc:.1%}")
print(f"Gap                            : {low_fill - low_mc:+.1%}")
print()
if low_fill - low_mc > 0.15:
    print("→ Substantial format effect. Fill-in answers carry materially less")
    print("  diagnostic signal. Categories weak on fill-in are UNDETECTABLE,")
    print("  not absent — report as a measurement limitation, not a finding")
    print("  about students.")
else:
    print("→ No large format effect. Low-frequency categories are plausibly")
    print("  genuinely rare in this curriculum rather than hidden by format.")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left — transfer comparison
cp = comp.set_index('Category')[['Eedi', 'FoundationalASSIST']] * 100
cp.plot(kind='barh', ax=axes[0], width=0.75)
axes[0].set_xlabel('Share of labeled items (%)')
axes[0].set_ylabel('')
axes[0].set_title('Category distribution: Eedi vs FoundationalASSIST', fontsize=11)
axes[0].axvline(2, color='crimson', ls='--', lw=1, label='2% viability floor')
axes[0].legend(fontsize=8)
axes[0].tick_params(labelsize=8)

# Right — confidence by format
(fmt_conf * 100).plot(kind='bar', stacked=True, ax=axes[1], width=0.6)
axes[1].set_ylabel('Share of items (%)')
axes[1].set_xlabel('')
axes[1].set_title('Labeling confidence by answer format', fontsize=11)
axes[1].tick_params(axis='x', rotation=0)
axes[1].legend(title='Confidence', fontsize=8)

plt.tight_layout()
plt.savefig('stage1b_pilot_findings.png', dpi=200, bbox_inches='tight')
plt.show()

### Low-confidence examples

Reading the actual cases the model was unsure about is the fastest way to see
whether the confidence signal is behaving sensibly.

In [ ]:
low = labeled[labeled['confidence'] == 'low']
print(f"{len(low)} low-confidence items ({len(low)/len(labeled):.1%})\n")
for _, r in low.head(8).iterrows():
    print(f"[{r['format']}] {r['category']}")
    print(f"   Q       : {textwrap.shorten(r['problem_text'], 110)}")
    print(f"   Correct : {str(r['correct'])[:40]}   Student: {str(r['wrong'])[:40]}")
    print(f"   Why     : {r['why']}\n")

## 11 · Decisions

Everything above feeds four decisions that must be settled before the LSTM
architecture is fixed, since together they determine the output dimension.

In [ ]:
results = {
    'generated_utc': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
    'n_pilot': int(len(labeled)),
    'model': LABELING_MODEL,
    'self_agreement': float(agree),
    'unassignable_share': float(unassign),
    'low_conf_fillin': float(low_fill),
    'low_conf_mc': float(low_mc),
    'distribution': {k: int(v) for k, v in dist.items()},
    'transfer': comp.to_dict('records'),
    'collapsed_categories': collapsed,
    'selectall_rows_masked': int(n_excluded),
}

with open('stage1b_pilot_results.json', 'w') as f:
    json.dump(results, f, indent=2)

print('DECISIONS REQUIRED')
print('=' * 76)
print(f"""
1. OUTPUT DIMENSION
   {len(category_names) - len(collapsed)} of {len(category_names)} categories cleared the 2% floor.
   Collapsed: {collapsed if collapsed else 'none'}
   -> Merge, drop, or retain with class weighting?

2. LABEL NOISE
   Self-agreement {agree:.1%} -> about {1 - agree:.1%} of training targets unstable.
   -> {'Acceptable; report the figure.' if agree > 0.9 else 'High. Consider majority vote over 3 passes.'}

3. FORMAT EFFECT
   Fill-in low-confidence {low_fill:.1%} vs MC {low_mc:.1%}.
   -> {'Report weak categories as undetectable, not absent.' if low_fill - low_mc > 0.15 else 'No strong format confound.'}

4. UNLABELED COVERAGE
   Two sources of unlabeled misconception rows:
     - select-all problems     : {n_excluded:,} rows ({n_excluded/605_773:.1%})
     - fill-in tail beyond top-{TOP_N_FILLIN} : roughly 26% of fill-in attempts
   -> Recommend masking both in the Stage 2 loss. They stay in the sequence as
      behavioural context (hints, timing, correctness) but contribute no gradient.
""")
print('=' * 76)
print('Saved stage1b_pilot_results.json + stage1b_pilot_findings.png')

if IN_COLAB:
    from google.colab import files
    files.download('stage1b_pilot_results.json')

---

## Next steps

1. **Review the low-confidence examples above.** If the model is unsure in cases
   where a human would also be unsure, the confidence signal is trustworthy and
   the format analysis holds.
2. **Settle the four decisions**, then record them in
   `capstone/capstone-project-design.md`.
3. **Scale to the full run** once the output dimension is fixed — roughly 20,520
   pairs at the natural 65/35 format split.
4. **Then** design the LSTM, with the output dimension already determined.

---

*Stage 1b pilot. Stage 1a produced the taxonomy; the full labeling run and
Stage 2 training follow.*